# Probing: do ModernBERT embeddings encode the CEFR vocabulary profile?

A ridge regression (linear probe) predicts the CEFR vocabulary proportions of
each text from its ModernBERT embedding, before and after fine-tuning.

- **Targets**: the six AW proportions (A1 to C2) and their mean word level
  (A1 = 1 ... C2 = 6, weighted by the proportions).
- **Representations**, each with CLS and mean pooling of the last layer:
  - `base`: pretrained ModernBERT, no fine-tuning
  - `fine-tuned`: ModernBERT fine-tuned for CEFR classification with the same
    settings as `hf_only` in `01_multiseed_training` (five seeds)
- **Baseline**: the gold CEFR level and text length (log word count) alone.
- **Measures** in the results table:
  - `R2`: how well the embedding alone predicts the target
  - `dR2`: what the embedding adds to level + length, i.e. the part that cannot
    be explained by the model simply knowing the proficiency level and length

The probe is fitted on the training set and evaluated on the held-out set:

- `STAGE = "val"`: validation set
- `STAGE = "final"`: test set

Embeddings are cached, so each model is only run once per stage.

In [1]:
import gc
from pathlib import Path

import numpy as np
import pandas as pd
import torch
from datasets import load_dataset
from sklearn.linear_model import RidgeCV
from sklearn.metrics import r2_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from transformers import (
    AutoModel,
    AutoTokenizer,
    DataCollatorWithPadding,
    ModernBertForSequenceClassification,
    Trainer,
    TrainingArguments,
    set_seed,
)

W1021 18:56:39.249000 31700 site-packages\torch\utils\_pytree.py:630] <enum 'KernelPreference'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.
W1021 18:56:39.347000 31700 site-packages\torch\utils\_pytree.py:630] <enum 'ScaleCalculationMode'> is an Enum subclass and is now natively supported by torch.compile as an opaque value type. Calling register_constant() on Enum subclasses is deprecated and will be an error in a future release.


In [2]:
# ---- Settings -------------------------------------------------------------
STAGE = "val"
EVAL = "val" if STAGE == "val" else "test"

DATA_DIR = Path(r"C:\Users\coope\Documents\Learner_Writing_Classification\cefr_lexical_sophistication\cefr-lexical-sophistication-main\wai_data\cefr_vocab_data")
OUT_DIR = Path("robustness_outputs")
EMB_DIR = OUT_DIR / f"embeddings_{STAGE}"
EMB_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "answerdotai/ModernBERT-base"
FT_SEEDS = [42, 1, 2, 3, 4]        # fine-tuned models (about 2 min each)
LEARNING_RATE = 1e-05              # same as 01
EPOCHS = 3
BATCH_SIZE = 8
VOCAB_MODE = "AW"

LABEL_MAP = {"A2": 0, "B1": 1, "B2": 2, "C1": 3}
LEVELS = ["A1", "A2", "B1", "B2", "C1", "C2"]
CEFR_KEYS = [f"cefr_{l}" for l in LEVELS]
TARGETS = LEVELS + ["mean_level"]

In [3]:
# ---- Load data (training set + held-out set only) ---------------------------
data = {}
for split in ["train", EVAL]:
    ds = load_dataset("json", data_files=str(DATA_DIR / f"{split}_cefr_{VOCAB_MODE}.jsonl"), split="train")
    props = np.array([[x[k] for k in CEFR_KEYS] for x in ds], dtype=np.float64)
    text = list(ds["text"])
    labels = np.array([LABEL_MAP[l] for l in ds["label"]])
    data[split] = {
        "text": text,
        "labels": labels,
        # targets: six proportions + mean word level (A1 = 1 ... C2 = 6)
        "targets": np.column_stack([props, props @ np.arange(1, 7)]),
        # baseline features: one-hot CEFR level + log word count
        "baseline": np.column_stack([np.eye(4)[labels], np.log([len(t.split()) for t in text])]),
    }
    print(split, len(text))

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
enc = {split: tokenizer(data[split]["text"], truncation=True, max_length=8192) for split in data}

class TextDataset(torch.utils.data.Dataset):
    def __init__(self, split):
        self.enc, self.labels = enc[split], data[split]["labels"]
    def __len__(self):
        return len(self.labels)
    def __getitem__(self, i):
        return {"input_ids": self.enc["input_ids"][i], "attention_mask": self.enc["attention_mask"][i],
                "labels": int(self.labels[i])}

collator = DataCollatorWithPadding(tokenizer=tokenizer)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

train 4195
val 401


In [4]:
# ---- Embedding extraction -----------------------------------------------------
@torch.no_grad()
def extract(encoder, split, batch_size=16):
    """CLS and mean-pooled last-layer embeddings for every text in a split."""
    encoder.eval().to(device)
    ds = TextDataset(split)
    cls, mean = [], []
    for start in range(0, len(ds), batch_size):
        batch = collator([ds[i] for i in range(start, min(start + batch_size, len(ds)))])
        ids, mask = batch["input_ids"].to(device), batch["attention_mask"].to(device)
        hidden = encoder(input_ids=ids, attention_mask=mask).last_hidden_state.float()
        cls.append(hidden[:, 0].cpu().numpy())
        m = mask.unsqueeze(-1).float()
        mean.append(((hidden * m).sum(1) / m.sum(1)).cpu().numpy())
    return np.concatenate(cls), np.concatenate(mean)

def save_embeddings(name, encoder):
    out = {}
    for split in ["train", EVAL]:
        out[f"{split}_cls"], out[f"{split}_mean"] = extract(encoder, split)
    np.savez(EMB_DIR / f"{name}.npz", **out)

In [5]:
# Base (pretrained) model
if not (EMB_DIR / "base.npz").exists():
    base = AutoModel.from_pretrained(MODEL_NAME)
    save_embeddings("base", base)
    del base
    gc.collect()
    torch.cuda.empty_cache()

Loading weights:   0%|          | 0/134 [00:00<?, ?it/s]

[transformers] ModernBertModel LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     |  | 
------------------+------------+--+-
decoder.bias      | UNEXPECTED |  | 
head.norm.weight  | UNEXPECTED |  | 
head.dense.weight | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


In [6]:
# Fine-tuned models: stock HF ModernBERT classifier (= hf_only in 01)
for seed in FT_SEEDS:
    if (EMB_DIR / f"finetuned_seed{seed}.npz").exists():
        continue
    set_seed(seed)
    model = ModernBertForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=4)
    args = TrainingArguments(output_dir=str(OUT_DIR / "tmp"), save_strategy="no", report_to="none",
                             seed=seed, data_seed=seed, learning_rate=LEARNING_RATE,
                             num_train_epochs=EPOCHS, per_device_train_batch_size=BATCH_SIZE,
                             logging_steps=200)
    Trainer(model=model, args=args, data_collator=collator, train_dataset=TextDataset("train")).train()
    save_embeddings(f"finetuned_seed{seed}", model.model)
    print(f"fine-tuned seed {seed} done")
    del model
    gc.collect()
    torch.cuda.empty_cache()

Loading weights:   0%|          | 0/136 [00:00<?, ?it/s]

[transformers] ModernBertForSequenceClassification LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
classifier.bias   | MISSING    | 
classifier.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Step,Training Loss
200,0.953165
400,0.740642
600,0.670776
800,0.555491
1000,0.558642
1200,0.463642
1400,0.402826


fine-tuned seed 42 done


Loading weights:   0%|          | 0/136 [00:00<?, ?it/s]

...

fine-tuned seed 3 done


Loading weights:   0%|          | 0/136 [00:00<?, ?it/s]

[transformers] ModernBertForSequenceClassification LOAD REPORT from: answerdotai/ModernBERT-base
Key               | Status     | 
------------------+------------+-
decoder.bias      | UNEXPECTED | 
classifier.bias   | MISSING    | 
classifier.weight | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Step,Training Loss
200,0.941558
400,0.833583
600,0.730549
800,0.651651
1000,0.650746
1200,0.544988
1400,0.511286


fine-tuned seed 4 done


In [7]:
# ---- Probes -----------------------------------------------------------------
# Ridge regression with the penalty chosen separately for each target by
# efficient leave-one-out CV on the training set. Embeddings are standardised.
ALPHAS = np.logspace(-1, 5, 13)

def probe_r2(X_train, X_eval):
    probe = make_pipeline(StandardScaler(), RidgeCV(alphas=ALPHAS, alpha_per_target=True))
    probe.fit(X_train, data["train"]["targets"])
    return r2_score(data[EVAL]["targets"], probe.predict(X_eval), multioutput="raw_values")

r2_baseline = probe_r2(data["train"]["baseline"], data[EVAL]["baseline"])

def probe_representation(emb_file, pooling):
    e = np.load(EMB_DIR / emb_file)
    Xtr, Xev = e[f"train_{pooling}"], e[f"{EVAL}_{pooling}"]
    r2_emb = probe_r2(Xtr, Xev)
    r2_both = probe_r2(np.hstack([Xtr, data["train"]["baseline"]]), np.hstack([Xev, data[EVAL]["baseline"]]))
    return r2_emb, r2_both - r2_baseline

rows = []
for pooling in ["cls", "mean"]:
    r2_emb, delta = probe_representation("base.npz", pooling)
    rows += [{"model": "base", "pooling": pooling, "seed": None, "measure": "R2", **dict(zip(TARGETS, r2_emb))},
             {"model": "base", "pooling": pooling, "seed": None, "measure": "dR2", **dict(zip(TARGETS, delta))}]
    for seed in FT_SEEDS:
        r2_emb, delta = probe_representation(f"finetuned_seed{seed}.npz", pooling)
        rows += [{"model": "fine-tuned", "pooling": pooling, "seed": seed, "measure": "R2", **dict(zip(TARGETS, r2_emb))},
                 {"model": "fine-tuned", "pooling": pooling, "seed": seed, "measure": "dR2", **dict(zip(TARGETS, delta))}]

per_seed = pd.DataFrame(rows)
per_seed.to_csv(OUT_DIR / f"{STAGE}_probe_per_seed.csv", index=False)

In [8]:
# ---- Results ------------------------------------------------------------------
# R2: embedding alone. dR2: added over CEFR level + length.
# Fine-tuned rows are mean (SD) over seeds.
fmt = lambda m, s: f"{m:.2f}" + ("" if np.isnan(s) else f" ({s:.2f})")
order = pd.MultiIndex.from_product([["R2", "dR2"], ["base", "fine-tuned"], ["cls", "mean"]],
                                   names=["measure", "model", "pooling"])
results = (per_seed.groupby(["measure", "model", "pooling"])[TARGETS]
           .agg(["mean", "std"]).reindex(order))
results = results.reorder_levels(["measure", "pooling", "model"]).sort_index(level=[0, 1], sort_remaining=False)
table = pd.DataFrame({t: [fmt(m, s) for m, s in zip(results[t]["mean"], results[t]["std"])] for t in TARGETS},
                     index=results.index)
baseline_row = pd.DataFrame([[f"{v:.2f}" for v in r2_baseline]], columns=TARGETS,
                            index=pd.MultiIndex.from_tuples([("R2", "", "level + length only")]))
table = pd.concat([baseline_row, table])
table.to_csv(OUT_DIR / f"{STAGE}_probe_results.csv")
print(f"Probe R2 on the {EVAL} set")
table

Probe R2 on the val set


A1           A2           B1  \
R2       level + length only         0.43        -0.06         0.31   
    cls  base                        0.67         0.11         0.52   
         fine-tuned           0.62 (0.08)  0.00 (0.06)  0.45 (0.08)   
    mean base                        0.80         0.34         0.57   
         fine-tuned           0.77 (0.01)  0.30 (0.02)  0.53 (0.02)   
dR2 cls  base                        0.24         0.17         0.21   
         fine-tuned           0.19 (0.07)  0.06 (0.06)  0.15 (0.05)   
    mean base                        0.36         0.40         0.26   
         fine-tuned           0.34 (0.01)  0.35 (0.02)  0.22 (0.02)   

                                       B2           C1           C2  \
R2       level + length only         0.32         0.11         0.19   
    cls  base                        0.62         0.26         0.35   
         fine-tuned           0.54 (0.07)  0.21 (0.05)  0.29 (0.04)   
    mean base                        0.68         0.31         0.35   
         fine-tuned           0.66 (0.02)  0.29 (0.04)  0.35 (0.03)   
dR2 cls  base                        0.31         0.15         0.16   
         fine-tuned           0.23 (0.05)  0.11 (0.05)  0.10 (0.04)   
    mean base                        0.36         0.21         0.16   
         fine-tuned           0.34 (0.02)  0.18 (0.04)  0.16 (0.03)   

                               mean_level  
R2       level + length only         0.46  
    cls  base                        0.79  
         fine-tuned           0.72 (0.09)  
    mean base                        0.87  
         fine-tuned           0.84 (0.01)  
dR2 cls  base                        0.34  
         fine-tuned           0.27 (0.07)  
    mean base                        0.41  
         fine-tuned           0.38 (0.01)